<a href="https://colab.research.google.com/github/Aniruddha-png/ECSR/blob/main/biometric_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 20000

# Generate timestamps
timestamps = pd.date_range("2025-01-01", periods=n, freq="5T")

# Simulate biometric signals with some time-series correlation
heart_rate = np.random.normal(80, 12, n).cumsum() / np.arange(1, n+1)
bp_sys = np.random.normal(120, 15, n).cumsum() / np.arange(1, n+1)
bp_dia = np.random.normal(80, 10, n).cumsum() / np.arange(1, n+1)
hrv = np.random.normal(45, 15, n).cumsum() / np.arange(1, n+1)
respiration_rate = np.random.normal(16, 3, n).cumsum() / np.arange(1, n+1)
gsr = np.random.normal(0.5, 0.2, n).cumsum() / np.arange(1, n+1)
skin_temp = np.random.normal(36.5, 0.5, n).cumsum() / np.arange(1, n+1)
body_temp = np.random.normal(37.0, 0.3, n).cumsum() / np.arange(1, n+1)

# Assemble dataframe
ts_data = pd.DataFrame({
    "timestamp": timestamps,
    "participant_id": np.arange(1, n+1),
    "heart_rate": heart_rate,
    "bp_sys": bp_sys,
    "bp_dia": bp_dia,
    "hrv": hrv,
    "respiration_rate": respiration_rate,
    "gsr": gsr,
    "skin_temp": skin_temp,
    "body_temp": body_temp
})

# Emotion labeling function
def label_emotion_ts(row):
    if row.heart_rate > 95 and row.bp_sys > 140 and row.respiration_rate > 20:
        return "anger"
    elif row.heart_rate < 65 and row.bp_sys < 110 and row.respiration_rate < 14:
        return "sad"
    elif 65 <= row.heart_rate <= 85 and row.hrv > 50 and row.gsr < 0.6:
        return "happy"
    elif 85 <= row.heart_rate <= 100 and row.gsr > 0.8 and row.skin_temp < 36.2:
        return "fear"
    elif row.heart_rate > 100 and row.respiration_rate > 22 and 0.7 <= row.gsr <= 0.9:
        return "surprise"
    elif 70 <= row.heart_rate <= 90 and (row.respiration_rate < 12 or row.respiration_rate > 20) and 0.6 <= row.gsr <= 0.8:
        return "disgust"
    else:
        return np.random.choice(["happy", "sad", "anger", "fear", "surprise", "disgust"])

ts_data["emotion"] = ts_data.apply(label_emotion_ts, axis=1)

# Shuffle to simulate multiple participants at different times
ts_data = ts_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to CSV
ts_data.to_csv("synthetic_biometric_6emotion_timeseries_20k.csv", index=False)

print(ts_data.head())


/tmp/ipython-input-3930916755.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  timestamps = pd.date_range("2025-01-01", periods=n, freq="5T")


            timestamp  participant_id  heart_rate      bp_sys     bp_dia  \
0 2025-02-06 23:30:00           10651   79.965856  119.869825  80.056931   
1 2025-01-08 02:05:00            2042   80.487545  119.987299  80.216993   
2 2025-01-31 02:20:00            8669   79.971200  119.865061  80.028330   
3 2025-01-04 20:50:00            1115   80.407471  119.687230  80.229366   
4 2025-02-18 06:30:00           13903   80.002247  119.865142  80.043279   

         hrv  respiration_rate       gsr  skin_temp  body_temp   emotion  
0  45.154955         16.021505  0.502179  36.496093  36.999771  surprise  
1  45.310283         16.022298  0.500279  36.483288  36.997754       sad  
2  45.212965         16.017408  0.502293  36.496278  36.998031  surprise  
3  45.482089         15.907669  0.502074  36.486920  36.996251  surprise  
4  45.108641         16.017046  0.502498  36.497991  36.999735       sad  


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization


In [ ]:
df = pd.read_csv("synthetic_biometric_6emotion_timeseries_20k.csv")
df.head()


,timestamp,participant_id,heart_rate,bp_sys,bp_dia,hrv,respiration_rate,gsr,skin_temp,body_temp,emotion
0,2025-02-06 23:30:00,10651,79.965856,119.869825,80.056931,45.154955,16.021505,0.502179,36.496093,36.999771,surprise
1,2025-01-08 02:05:00,2042,80.487545,119.987299,80.216993,45.310283,16.022298,0.500279,36.483288,36.997754,sad
2,2025-01-31 02:20:00,8669,79.971200,119.865061,80.028330,45.212965,16.017408,0.502293,36.496278,36.998031,surprise
3,2025-01-04 20:50:00,1115,80.407471,119.687230,80.229366,45.482089,15.907669,0.502074,36.486920,36.996251,surprise
4,2025-02-18 06:30:00,13903,80.002247,119.865142,80.043279,45.108641,16.017046,0.502498,36.497991,36.999735,sad


In [ ]:
df.sort_values(by=["participant_id", "timestamp"], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()


,timestamp,participant_id,heart_rate,bp_sys,bp_dia,hrv,respiration_rate,gsr,skin_temp,body_temp,emotion
0,2025-01-01 00:00:00,1,85.960570,125.224294,75.281425,46.182566,17.992003,0.706119,36.390675,36.920339,fear
1,2025-01-01 00:05:00,2,82.150699,124.737074,82.704225,49.813014,16.551828,0.487524,36.541072,36.831479,happy
2,2025-01-01 00:10:00,3,84.024554,118.475450,81.142194,49.914186,16.301299,0.530045,36.649905,36.932390,fear
3,2025-01-01 00:15:00,4,87.587505,121.030028,81.083068,43.893461,16.094437,0.491572,36.464508,36.873195,sad
4,2025-01-01 00:20:00,5,85.508036,116.353775,82.301236,43.556108,16.383982,0.480161,36.481680,36.966616,surprise


In [ ]:
# Select features
features = ["heart_rate", "bp_sys", "bp_dia", "hrv", "respiration_rate", "gsr", "skin_temp", "body_temp"]
X = df[features].values
y = df["emotion"].values

# Encode emotions
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)


Classes: ['anger' 'disgust' 'fear' 'happy' 'sad' 'surprise']


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)


Train shape: (16000, 8) (16000,)
Validation shape: (4000, 8) (4000,)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


In [ ]:
seq_len = 10

def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len-1])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, seq_len)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val, seq_len)

print("Train seq shape:", X_train_seq.shape, y_train_seq.shape)
print("Val seq shape:", X_val_seq.shape, y_val_seq.shape)


Train seq shape: (15990, 10, 8) (15990,)
Val seq shape: (3990, 10, 8) (3990,)


In [ ]:
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(seq_len, len(features))))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dense(6, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 8, 64)          │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,318 (427.02 KB)

 Trainable params: 109,190 (426.52 KB)

 Non-trainable params: 128 (512.00 B)

In [ ]:
history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30,
    batch_size=64
)


Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.1678 - loss: 1.7988 - val_accuracy: 0.1642 - val_loss: 1.7927
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1844 - loss: 1.7883 - val_accuracy: 0.1734 - val_loss: 1.7985
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1897 - loss: 1.7851 - val_accuracy: 0.1609 - val_loss: 1.7963
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1964 - loss: 1.7823 - val_accuracy: 0.1732 - val_loss: 1.7957
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.2070 - loss: 1.7768 - val_accuracy: 0.1692 - val_loss: 1.7989
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.2169 - loss: 1.7718 - val_accuracy: 0.1739 - val_loss: 1.8026
Epoch 7/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.2201 - loss: 1.7657 - val_accuracy: 0.1692 - val_loss: 1.8048
Epoch 8/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.2230 - loss: 1.7590 - val_accuracy: 